# soundai — クイックスタート

このノートブックでは次の3ステップを試します。

1. `data/samples/` の音源をスキャンして CLAP embedding を生成
2. 自然言語クエリで検索
3. 音源ファイルで類似検索

> **前提**: `data/samples/` に WAV / AIFF / MP3 ファイルを配置してください。

In [ ]:
import sys
sys.path.insert(0, "..")

from pathlib import Path
from tqdm.notebook import tqdm

from soundai.embed import CLAPEmbedder, SUPPORTED_EXTENSIONS
from soundai.index import EmbeddingIndex
from soundai.search import SampleSearcher

SAMPLES_DIR = Path("../data/samples")
INDEX_DIR   = Path("../data/embeddings")

## 1. 音源をインデックスに登録

CLI からも実行できます:
```bash
uv run python scripts/ingest.py
```

In [ ]:
# 音源ファイルを収集
audio_files = [
    p for p in sorted(SAMPLES_DIR.rglob("*"))
    if p.is_file() and p.suffix.lower() in SUPPORTED_EXTENSIONS
]
print(f"見つかったファイル数: {len(audio_files)}")
audio_files[:5]

In [ ]:
# Embedding 生成 (初回は CLAP モデルのダウンロードが走ります ~400MB)
BATCH_SIZE = 32

embedder = CLAPEmbedder()
index    = EmbeddingIndex()

for i in tqdm(range(0, len(audio_files), BATCH_SIZE)):
    batch = audio_files[i : i + BATCH_SIZE]
    embs  = embedder.embed_files(batch)
    if embs.shape[0] > 0:
        index.add(embs, [str(p) for p in batch[:embs.shape[0]]])

index.save(INDEX_DIR)
print(f"インデックス完成: {index.size} 件")

## 2. 自然言語で検索

In [ ]:
searcher = SampleSearcher(index_dir=INDEX_DIR)

query = "deep 808 bass kick"   # ← ここを自由に変えてみてください

results = searcher.search_by_text(query, k=10)
searcher.pretty_print(results)

## 3. 音源ファイルで類似検索

In [ ]:
# 手持ちのファイルパスを指定してください
example_file = audio_files[0] if audio_files else None

if example_file:
    print(f"クエリ: {example_file.name}")
    results = searcher.search_by_file(example_file, k=10)
    searcher.pretty_print(results)
else:
    print("data/samples/ に音源ファイルを配置してください")

## 4. (ボーナス) UMAP で埋め込み空間を可視化

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import umap

# インデックスから全 embedding を取り出す
n = index.size
all_embs = np.zeros((n, index.dim), dtype="float32")
index._index.reconstruct_n(0, n, all_embs)

reducer = umap.UMAP(n_components=2, random_state=42)
coords  = reducer.fit_transform(all_embs)

labels = [Path(p).parent.name for p in index._paths]   # フォルダ名をラベルに

plt.figure(figsize=(10, 8))
unique_labels = list(set(labels))
cmap = plt.cm.get_cmap("tab20", len(unique_labels))
for i, lbl in enumerate(unique_labels):
    mask = [l == lbl for l in labels]
    plt.scatter(coords[mask, 0], coords[mask, 1],
                label=lbl, s=20, alpha=0.7, color=cmap(i))
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
plt.title("Audio Embedding Space (UMAP)")
plt.tight_layout()
plt.show()